In [1]:
import os
import sys
os.chdir('..')
sys.path.insert(0, '.')


import pandas as pd
from pathlib import Path
from retrieval.bm25_index import search as bm25_search
from retrieval.vector_store import search as vector_search
from retrieval.hybrid import search as hybrid_search

print('Imports OK.')

Imports OK.


In [2]:
TEST_QUESTIONS = [
    {'id': 'q01', 'question': 'BERTurk ne zaman yayınlandı?',                                    'relevant_sources': ['BERTURK_README_cleaned']},
    {'id': 'q02', 'question': 'AeroGuard sisteminin doğruluk oranı nedir?',                      'relevant_sources': ['AeroGuard_README_cleaned']},
    {'id': 'q03', 'question': 'AeroGuard hangi havalimanlarında kullanılıyor?',                  'relevant_sources': ['AeroGuard_README_cleaned']},
    {'id': 'q04', 'question': 'Attention mekanizması transformer mimarisinde nasıl çalışır?',    'relevant_sources': ['AttentionIsAllYouNeed_cleaned']},
    {'id': 'q05', 'question': 'THQuAD veri seti ne için kullanılır?',                            'relevant_sources': ['THQuAD_cleaned']},
    {'id': 'q06', 'question': 'LangChain nedir ve ne işe yarar?',                                'relevant_sources': ['LANGCHAIN_README_cleaned']},
    {'id': 'q07', 'question': 'LangGraph state graph nasıl tanımlanır?',                         'relevant_sources': ['LANGGRAPH_README_cleaned']},
    {'id': 'q08', 'question': 'Multimodal dil modelleri görüntü ve metni nasıl birleştirir?',   'relevant_sources': ['Multimodal_LLM_Paper_cleaned']},
    {'id': 'q09', 'question': 'DSL ve HFC erişim ağları arasındaki fark nedir?',                 'relevant_sources': ['AdvancedNetworkNotlarım_cleaned']},
    {'id': 'q10', 'question': 'Yapay zeka eğitiminde fine-tuning nedir?',                        'relevant_sources': ['IntroToAINotlarım_cleaned']},
]

print(f'Loaded {len(TEST_QUESTIONS)} test questions.')

Loaded 10 test questions.


In [3]:
def precision_at_k(results: list, relevant_sources: list, k: int = 5) -> float:
    """Calculating precision@k — relevant results in top-k / k."""
    top_k = results[:k]
    hits  = sum(1 for r in top_k if r['source'] in relevant_sources)
    return round(hits / k, 3)


def recall_at_k(results: list, relevant_sources: list, k: int = 5) -> float:
    """Calculating recall@k — unique relevant sources found in top-k / total relevant."""
    top_k         = results[:k]
    found_sources = set(r['source'] for r in top_k if r['source'] in relevant_sources)
    return round(len(found_sources) / len(relevant_sources), 3)


print('Evaluation functions defined.')

Evaluation functions defined.


In [4]:
# Running retrieval benchmark across all methods
rows = []

for q in TEST_QUESTIONS:
    bm25_res   = bm25_search(q['question'],   top_k=5)
    dense_res  = vector_search(q['question'], top_k=5)
    hybrid_res = hybrid_search(q['question'], top_k=5)

    rows.append({
        'id':          q['id'],
        'question':    q['question'][:50],
        'bm25_p@5':    precision_at_k(bm25_res,   q['relevant_sources']),
        'bm25_r@5':    recall_at_k(bm25_res,      q['relevant_sources']),
        'dense_p@5':   precision_at_k(dense_res,  q['relevant_sources']),
        'dense_r@5':   recall_at_k(dense_res,     q['relevant_sources']),
        'hybrid_p@5':  precision_at_k(hybrid_res, q['relevant_sources']),
        'hybrid_r@5':  recall_at_k(hybrid_res,    q['relevant_sources']),
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

Loading chunks from AdvancedNetworkNotlarım_cleaned.txt...
  Generated 89 chunks.
Loading chunks from AeroGuard_README_cleaned.txt...
  Generated 3 chunks.
Loading chunks from AttentionIsAllYouNeed_cleaned.txt...
  Generated 2 chunks.
Loading chunks from BERTURK_README_cleaned.txt...
  Generated 1 chunks.
Loading chunks from IntroToAINotlarım_cleaned.txt...
  Generated 138 chunks.
Loading chunks from LANGCHAIN_README_cleaned.txt...
  Generated 2 chunks.
Loading chunks from LANGGRAPH_README_cleaned.txt...
  Generated 1 chunks.
Loading chunks from Multimodal_LLM_Paper_cleaned.txt...
  Generated 6 chunks.
Loading chunks from THQuAD_cleaned.txt...
  Generated 7 chunks.
Loading complete — 249 total chunks.
Building BM25 index...
Building complete — 249 chunks indexed.
Initializing ChromaDB collection at /Users/burakegekaya/Desktop/AgentBench-TR/storage/chroma...
Collection 'agentbench_tr' ready — 249 docs stored.
 id                                           question  bm25_p@5  bm25_r@5  de

In [5]:
print('=== AVERAGES ===')
print(f'BM25   — Precision@5: {df["bm25_p@5"].mean():.3f} | Recall@5: {df["bm25_r@5"].mean():.3f}')
print(f'Dense  — Precision@5: {df["dense_p@5"].mean():.3f} | Recall@5: {df["dense_r@5"].mean():.3f}')
print(f'Hybrid — Precision@5: {df["hybrid_p@5"].mean():.3f} | Recall@5: {df["hybrid_r@5"].mean():.3f}')

Path('results').mkdir(exist_ok=True)
df.to_csv('results/retrieval_benchmark.csv', index=False)
print('\nSaving results to results/retrieval_benchmark.csv — done.')

=== AVERAGES ===
BM25   — Precision@5: 0.260 | Recall@5: 0.800
Dense  — Precision@5: 0.380 | Recall@5: 0.800
Hybrid — Precision@5: 0.380 | Recall@5: 1.000

Saving results to results/retrieval_benchmark.csv — done.
